In [ ]:
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view

np.random.seed(0)

# Stationarity

- Mean is constant
- Variance is constant
- No Seasonality

## Isnt this just White Noise?

While the above conditions look very similar to white noise, there is a subtle difference. 
- Stationarity: mean is constant
- White noise: mean is 0

This is illustrated in the plot below.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(6, 3), sharey=True)

n = 1000
x = np.arange(n)

white_noise = np.random.normal(0, 1, n)
axs[0].plot(x, white_noise)
axs[0].set_title("White noise $\\mu=0$")
axs[0].hlines(0, 0, n, color="orange", linestyle="dashed", label="Mean")

stationarity = np.random.normal(5, 1, n)
axs[1].plot(x, stationarity)
axs[1].set_title("Stationary $\\mu\\ne0$")
axs[1].hlines(5, 0, n, color="orange", linestyle="dashed", label="Mean")

fig.tight_layout()
plt.legend(bbox_to_anchor=(1.4, 1))

## Violating Stationarity Assumptions

To further illustrate the concept of stationarity, let's examine three examples where one of the key assumptions is intentionally violated, while the others are held constant:
- A series with constant mean and seasonality, but changing variance
- A series with changing mean (trend), but constant variance and no seasonality
- A series with constant mean and variance, but seasonal structure

In [ ]:
# helper function
def plot_series_with_rolling_stats(x, y, window=50, title="Time Series", figsize=(8, 5)):
    """
    Plot a time series along with its rolling mean and rolling variance.
    """
    # Rolling statistics
    rolling_mean = sliding_window_view(y, window).mean(axis=1)
    rolling_var = sliding_window_view(y, window).var(axis=1)

    # Pad to match x axis
    pad = [np.nan] * (window - 1)
    mean_padded = np.concatenate([pad, rolling_mean])
    var_padded = np.concatenate([pad, rolling_var])

    # Plot
    fig, axes = plt.subplots(3, 1, figsize=figsize, sharex=True)

    # Original series
    axes[0].plot(x, y, color="blue", alpha=0.7)
    axes[0].axhline(0, color='black', linestyle='--', linewidth=1)
    axes[0].set_title(f"{title}")

    # Rolling mean
    axes[1].plot(x, y, color="blue", alpha=0.3)
    axes[1].plot(x, mean_padded, color="red")
    axes[1].set_title(f"Rolling Mean (window={window})")

    # Rolling variance
    axes[2].plot(x, y, color="blue", alpha=0.3)
    axes[2].plot(x, var_padded, color="green")
    axes[2].set_title(f"Rolling Variance (window={window})")
    axes[2].set_xlabel("Time")

    plt.tight_layout()
    plt.show()

In [ ]:
x = np.linspace(0, 100, 1000)
scales = np.linspace(3, 0.5, len(x))
y1 = np.random.normal(loc=0, scale=scales)
plot_series_with_rolling_stats(x, y1, title="1: Constant Mean, Changing Variance")

In [ ]:
# changing mean, constant variance
# these coeffs are purely arbitary, and took me way too long to find out
y2 = np.polyval([3e-04, -6e-02, 3.2e+00, -2e+01], x) + np.random.normal(0, 2, len(x))
plot_series_with_rolling_stats(x, y2, title="2: Changing Mean, Constant Variance")

In [ ]:
y3 = 1 * np.sin(2 * np.pi * x / 15) + np.random.normal(0, 0.5, len(x))
# as the period is 15, increase window size to show constant mean
plot_series_with_rolling_stats(x, y3, title="3: Seasonality", window=150)

## Detecting Trends and Seasonality

### Autocorrelation Function (ACF) and Partial Autocorrelation Function (PACF)

Autocorrelation is the measure of correlation between a time series, and a lagged version of itself. Given a time series $ y_t$, and some lag of $ k $, the autocorrelation of $ y_t $ can be defined as:
$$ Corr(y_t,\ y_{t-k}) $$
More formally, the autocorrelation coefficient at lag $k$ is often represented as:
$$\rho_k = \frac{Cov(y_t, y_{t-k})}{\sqrt{Var(y_t)Var(y_{t-k})}}$$
This tells us how well the past values of the series predict the present, based on linear relationships.

**It's important to note that the clear interpretations of ACF and PACF discussed below are most directly applicable to *stationary* time series.** 

This section aims to provide intuition behind the two main *flavours* of autocorrelation:
- Autocorrelation Function (ACF)
- Partial Autocorrelation Function (PACF)

Let's first set up an example time-series:
$$ y_t,\ y_{t-1},\ y_{t-2},\ ... $$
To provide a more concrete example, let's use months as the time:
$$ y_{Mar},\ y_{Feb},\ y_{Jan},\ ... $$
Now suppose this time-series represented stock prices, or temperature; it would be reasonable to assume that:
- *January's* value effects *February's*
- *February's* value effects *March's*
- And so on...

This is demonstrated below:

```mermaid
flowchart LR

A(Jan) --> |Effects| B(Feb)
B --> |Effects| C(Mar)
```

However, it is often that in real-world data past values influence each other recursively, that is:

```mermaid
flowchart LR

A --> |Effects| C
A(Jan) --> |Effects| B(Feb)
B --> |Effects| C(Mar)
```

Now let's suppose that we are interested in the correlation at lag $ k=2 $:
$$ Corr(y_{Mar},\ y_{Jan}) $$
We could simply calculate the Pearson's correlation between $ y_{Mar} $ and $ y_{Jan} $ , $ y_{Apr} $ and $ y_{Feb} $ , $ y_{May} $ and $ y_{Mar} $ and so on... However, this would capture both of the following relationships:
- Direct influence of $ Jan \rightarrow Mar $
- Indirect influence of $ Jan \rightarrow Feb \rightarrow Mar $

What if we were *only* interested in the direct influnce *Jan* has on *Mar*. This brings us to the distinction between ACF and PACF:
- ACF includes *all* direct and indirect relationships between the original series, and it's lagged version
- PACF allows us to investigate the relationships in isolation, effectively removing the influence of intermediate lags.

#### ACF: Example

Much like we have done previously, let's generate a noisy sine signal to use as an example. Note, we create the signal with 100 data points, and a period of 20. We should expect to see 5 peaks.

In [ ]:
n = 100
t = np.arange(n)

period = 20
y = 1 * np.sin(2 * np.pi * t / period) + np.random.normal(0, 0.5, n)

plt.figure(figsize=(6, 3))
plt.plot(t, y, label='Time Series')
plt.title('Synthetic Time Series')
plt.xlabel('Time')
plt.ylabel('Value')
plt.legend()
plt.show()

As expected, there are 5 clear peaks in the data. 
Let's now perform **ACF**, up to $ k=50 $ (50 lags). We shall use `scipy`'s `plot_acf` function, as it also generates a plot from the results.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

plt.figure(figsize=(8, 5))
plot_acf(y, lags=50) # Plot ACF up to 50 lags
plt.title('Autocorrelation')
plt.xlabel('Lag')
plt.ylabel('Autocorrelation Coefficient')

### Fast Fourier Transform (FFT)

## Statistial Tests for Stationarity

### Augmented Dickey-Fuller (ADF)

### Kwiatkowski-Phillips-Schmidt-Shin (KPSS)

## Converting to Stationary

### Overview